<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">
  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">
  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">Trabajo de Fin de Máster</div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">Clasificador taxonómico de boletines oficiales españoles</div>
  </div>
  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>
  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** - ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** - Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?** - Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** - Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

### Parte I - Fundamentos
|   | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo local |
| **1** | **Schema de output** | `ClassifierOutput`, enums e invariantes |
| **2** | **Pre-procesamiento** | N0 lookup + N1 clasificador por reglas |
| **3** | **Ground truth** | Muestreo estratificado + anotación manual |
| **4** | **Agente base** | System prompt, construcción y casos cualitativos |

### Parte II - Ciclo de experimentación
|   | Sección | Contenido |
|---|---------|----------|
| **5** | **Experimento 1 - Baseline** | Zero-shot, sin contexto N1 |
| **6** | **Análisis de errores** | Qué falla y por qué |
| **7** | **Prompt v2** | Mejora basada en errores (DEC-013 a DEC-019) |
| **8** | **Experimento 2 - Prompt v2** | ¿Mejora respecto a baseline? |
| **9** | **Experimento 3 - Ablación +N1** | ¿Aporta el contexto de forma? |
| **10** | **Experimento 4 - Few-shot** | ¿Ayudan los ejemplos reales? |
| **11** | **Comparativa de modelos** | Qwen 3.5 9B vs Gemma 4 4B |

### Parte III - Análisis final
|   | Sección | Contenido |
|---|---------|----------|
| **12** | **Tabla resumen** | Comparativa de todos los experimentos |
| **13** | **Calibración de confianza** | ¿El modelo sabe cuándo no sabe? |
| **14** | **Conclusiones y trabajo futuro** | Hallazgos, limitaciones, v2 |

---

##  0. Setup

Cargamos las variables de entorno e importamos las librerías. El modelo vive en LM Studio - el único punto de cambio para conectar otro proveedor es `LM_STUDIO_MODEL`.

In [9]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

load_dotenv(find_dotenv())

True

In [10]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# ── Modelo local via LM Studio ────────────────────────────────────────────────
# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

print(f"Modelo: {LM_STUDIO_MODEL} via LM Studio (localhost:1234)")

Modelo: qwen/qwen3.5-9b via LM Studio (localhost:1234)


/tmp/ipykernel_1805380/3953150006.py:8: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  model = OpenAIModel(


In [ ]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    print(f"LM Studio operativo.")
    print(f"Modelos disponibles: {modelos}")
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado"
    print(f"✓ {LM_STUDIO_MODEL} listo")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - ¿está el servidor arrancado?")

LM Studio operativo.
Modelos disponibles: ['google/gemma-4-e4b', 'qwen/qwen3.5-9b', 'google/gemma-3n-e4b', 'gemma-4-e2b-it', 'google/gemma-3-4b', 'text-embedding-nomic-embed-text-v1.5']
✓ qwen/qwen3.5-9b listo


---

##  1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) - resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) - multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) - multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [12]:
from clasificador.schema import ActType, ProcedureType, TechnologyType, ClassifierOutput

In [13]:
# Verificación de invariantes
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning="'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False, act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP], technologies=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError → {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "resolución",
  "procedures": [
    "DIA"
  ],
  "technologies": [
    "fotovoltaica"
  ],
  "confidence": 0.98,
  "reasoning": "'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica."
}

Violación de invariante:
  ValidationError → Value error, is_relevant=False con procedures != []


---

##  2. Pre-procesamiento

Antes de llamar al LLM, cada registro pasa por dos pasos deterministas:

- **N0 - Ámbito**: lookup directo del campo `bulletin` → `estatal / autonómico / local`. Sin LLM.
- **N1 - Tipo de acto**: clasificador de primer token con pre-procesamiento de formatos especiales (BOCM, BOCA, BOE topónimos).

**Cobertura real medida**: 88.6% del corpus. El 11.4% restante cae en `OTROS` - principalmente topónimos BOE irrecuperables sin PDF.

In [14]:
PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"

df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(html.unescape)

print(f"Corpus: {len(df):,} registros · {df['bulletin'].nunique()} boletines")

Corpus: 65,201 registros · 19 boletines


In [ ]:
# N0: ámbito por bulletin 
_GAZETTE_TO_AMBITO = {
    "boe": "estatal",
    "madridambiental": "local",
    # resto → autonómico por defecto
}

def get_ambito(bulletin: str) -> str:
    return _GAZETTE_TO_AMBITO.get(bulletin.lower(), "autonómico")


# N1: tipo de acto por primer token 
def preprocess_description(desc: str, bulletin: str) -> str:
    desc = desc.strip()
    if "\n–" in desc:
        desc = desc.split("\n–", 1)[1].strip()
    elif "\n-" in desc:
        desc = desc.split("\n-", 1)[1].strip()
    if bulletin.lower() in ("boca", "boib") and ".-" in desc:
        desc = desc.split(".-", 1)[1].strip()
    if re.match(r"^[A-ZÁÉÍÓÚÜÑ/\s]+$", desc) and len(desc.split()) <= 4:
        return "__TOPONIMO__"
    if desc.upper().startswith(("U.R.", "E.R.", "SUMA GESTIÓN", "ORGANISMO AUTÓNOMO DE HACIENDA")):
        return "__SUBASTA_AEAT__"
    return desc


_N1_MAP = [
    (r"corrección de errat",     ActType.CORRECCION_ERRORES),
    (r"corrección de error",     ActType.CORRECCION_ERRORES),
    (r"rectificación",           ActType.CORRECCION_ERRORES),
    (r"real decreto",            ActType.REAL_DECRETO),
    (r"orden foral",             ActType.ORDEN),
    (r"información pública",     ActType.INFORMACION_PUBLICA),
    (r"exposición pública",      ActType.INFORMACION_PUBLICA),
    (r"trámite de información",  ActType.INFORMACION_PUBLICA),
    (r"resolución",              ActType.RESOLUCION),
    (r"anuncio",                 ActType.ANUNCIO),
    (r"orden",                   ActType.ORDEN),
    (r"decreto foral",           ActType.DECRETO),
    (r"decreto",                 ActType.DECRETO),
    (r"acuerdo",                 ActType.ACUERDO),
    (r"aprobación",              ActType.APROBACION),
    (r"extracto",                ActType.EXTRACTO),
    (r"convenio",                ActType.CONVENIO),
    (r"adenda",                  ActType.CONVENIO),
    (r"solicitud",               ActType.SOLICITUD),
    (r"modificación",            ActType.MODIFICACION),
    (r"edicto",                  ActType.EDICTO),
    (r"notificación",            ActType.NOTIFICACION),
    (r"notificaciones",          ActType.NOTIFICACION),
    (r"recaudación ejecutiva",   ActType.NOTIFICACION),
    (r"propuesta de resolución", ActType.RESOLUCION),
    (r"bases",                   ActType.CONVOCATORIA),
    (r"convocatoria",            ActType.CONVOCATORIA),
    (r"nombramiento",            ActType.RESOLUCION),
    (r"delegación",              ActType.RESOLUCION),
    (r"emplazamiento",           ActType.NOTIFICACION),
    (r"citación",                ActType.NOTIFICACION),
    (r"diligencia",              ActType.NOTIFICACION),
    (r"cédula",                  ActType.NOTIFICACION),
    (r"requerimiento",           ActType.NOTIFICACION),
    (r"sala primera",            ActType.OTROS),
    (r"sala segunda",            ActType.OTROS),
    (r"__toponimo__",            ActType.OTROS),
    (r"__subasta_aeat__",        ActType.OTROS),
    (r"concesión",               ActType.RESOLUCION),
    (r"trámite de audiencia",    ActType.INFORMACION_PUBLICA),
    (r"trámite de",              ActType.INFORMACION_PUBLICA),
    (r"iniciación",              ActType.RESOLUCION),
    (r"inicio",                  ActType.RESOLUCION),
    (r"apertura",                ActType.RESOLUCION),
    (r"informe",                 ActType.RESOLUCION),
    (r"expediente",              ActType.RESOLUCION),
    (r"ley",                     ActType.OTROS),        # disposiciones normativas
    (r"recurso",                 ActType.RESOLUCION),   # recursos administrativos
    (r"notaría",                 ActType.RESOLUCION),   # actas notariales BON
    (r"publicación",             ActType.ANUNCIO),      # anuncios de publicación
    (r"plan",                    ActType.APROBACION),   # planes aprobados
    (r"departamento",            ActType.RESOLUCION),   # BOIB residual
]


def inferir_act_type(description: str, bulletin: str) -> ActType:
    desc_clean = preprocess_description(description, bulletin)
    text = desc_clean.lower().strip()
    for pattern, act_type in _N1_MAP:
        if text.startswith(pattern):
            return act_type
    return ActType.OTROS

In [16]:
df["act_type_n1"] = df.apply(
    lambda row: inferir_act_type(row["description"], row["bulletin"]), axis=1
)

dist = df["act_type_n1"].value_counts()
total = len(df)
print("Distribución N1 inferida:\n")
for val, count in dist.items():
    print(f"  {val:<25} {count:>6,}  ({count/total*100:.1f}%)")

otros = (df["act_type_n1"] == ActType.OTROS).sum()
print(f"\nCobertura N1: {(1 - otros/total)*100:.1f}%  ({otros:,} en OTROS)")

Distribución N1 inferida:

  resolución                23,849  (36.6%)
  anuncio                   16,382  (25.1%)
  otros                      6,752  (10.4%)
  orden                      3,198  (4.9%)
  aprobación                 2,930  (4.5%)
  notificación               2,565  (3.9%)
  información_pública        1,589  (2.4%)
  edicto                     1,486  (2.3%)
  extracto                   1,386  (2.1%)
  acuerdo                    1,321  (2.0%)
  corrección_errores           954  (1.5%)
  decreto                      808  (1.2%)
  convocatoria                 761  (1.2%)
  convenio                     686  (1.1%)
  real_decreto                 251  (0.4%)
  solicitud                    173  (0.3%)
  modificación                 110  (0.2%)

Cobertura N1: 89.6%  (6,752 en OTROS)


### Límites del clasificador N1 y trabajo futuro

El clasificador de primer token cubre **89.6%** del corpus con reglas deterministas.
El 10.4% restante cae en `OTROS` por tres motivos con soluciones conocidas:

| Grupo | Volumen aprox. | Motivo | Solución futura |
|---|---|---|---|
| Topónimos BOE / subastas AEAT | ~5.200 | Sin contenido textual real - irrecuperable sin PDF | Clase propia `NO_INFERIBLE` en v2 |
| BON fiscal | ~160 | `tipos`, `calendario` - formatos tributarios navarros | Reglas específicas BON |
| RRHH sin tipo explícito | ~350 | `relación`, `lista`, `oferta`, `bajas` - tipo de acto no en la descripción | LLM zero-shot viable en v2 (DEC-011) |
| BOIB residual | ~300 | Variantes de prefijo catalán no contempladas | Ampliar split `.-` para BOIB |
| Long tail distribuido | ~740 | Tokens poco frecuentes sin patrón claro | Cobertura ~98.5% teórica con PDF |

**Techo práctico estimado sin PDF**: ~98.5% añadiendo más tokens al mapa.
**Techo real con PDF**: ~100% - el tipo de acto aparece siempre en el cuerpo del documento.

---

##  3. Ground truth

El ground truth se construye en tres capas:

| Capa | Qué etiqueta | Cómo | 
|------|-------------|------|
| **1 - N1 determinista** | `act_type` | Reglas de primer token ( 2) | 
| **2 - Muestreo estratificado** | Selección de 100 registros | Keywords por procedimiento N2 | 
| **3 - Anotación manual** | `is_relevant_gt`, `procedures_gt`, `technologies_gt` | Revisión humana |

El archivo `ground_truth_100_anotado.csv` contiene los 100 registros con etiquetas manuales validadas.

In [ ]:
import random
random.seed(42)

# Paso 1: muestreo estratificado 500 registros 
keywords = {
    "DIA":     ["declaración de impacto ambiental"],
    "AAP":     ["autorización administrativa previa"],
    "AAC":     ["autorización de construcción", "autorización administrativa de construcción"],
    "AAP_AAC": ["previa y de construcción"],
    "AAU":     ["autorización ambiental unificada"],
    "IIA":     ["informe de impacto ambiental"],
    "AAI":     ["autorización ambiental integrada"],
    "IAE":     ["ambiental estratégic"],
    "DUP":     ["utilidad pública"],
}
cuotas = {"DIA": 60, "AAP": 60, "AAC": 50, "AAP_AAC": 40,
          "AAU": 40, "IIA": 40, "AAI": 30, "IAE": 30, "DUP": 30}
N_NEGATIVOS = 120

sampled_ids = set()
frames = []
for grupo, kws in keywords.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas[grupo], len(pool))
    sample = pool.sample(n, random_state=42).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<10} pool={len(pool):>5,}  sampled={n}")

all_kws = [kw for kws in keywords.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=42).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_gt_full = pd.concat(frames, ignore_index=True)
df_gt_full["id"] = range(len(df_gt_full))
print(f"\nPaso 1 - {len(df_gt_full)} registros muestreados")
print(df_gt_full["grupo_muestreo"].value_counts().to_string())

# ── Paso 2: submuestra proporcional de 100 registros ─────────────────────────
total = len(df_gt_full)
df_gt_100 = pd.concat([
    g.sample(min(len(g), max(1, round(len(g) * 100 / total))), random_state=42)
    for _, g in df_gt_full.groupby("grupo_muestreo")
]).reset_index(drop=True)

print(f"\nPaso 2 - {len(df_gt_100)} registros en la submuestra:")
print(df_gt_100["grupo_muestreo"].value_counts().to_string())


  DIA        pool=  286  sampled=60
  AAP        pool=1,016  sampled=60
  AAC        pool=  514  sampled=50
  AAP_AAC    pool=  243  sampled=40
  AAU        pool=  182  sampled=40
  IIA        pool=  521  sampled=40
  AAI        pool=  257  sampled=30
  IAE        pool=  319  sampled=30
  DUP        pool=  635  sampled=30

Total muestreado: 500 registros


### Dataset anotado

Los 100 registros del ground truth han sido anotados manualmente. Durante el proceso se identificaron casos especiales documentados en `decisiones_implementacion.md`:

- **Denegaciones**: heredan el tipo de procedimiento del acto denegado
- **Modificaciones**: heredan los procedimientos del acto modificado
- **Falsos positivos**: RRHH con vocabulario ambiental, concesiones de dominio público no energéticas

In [ ]:
#  Paso 3: cargar el CSV anotado manualmente 
PATH_GT_ANOTADO = "../data/ground_truth/ground_truth_100_anotado.csv"
df_anotado = pd.read_csv(PATH_GT_ANOTADO)

print(f"Ground truth anotado: {len(df_anotado)} registros")
print(f"Relevantes:     {df_anotado['is_relevant_gt'].astype(bool).sum()}")
print(f"No relevantes:  {(~df_anotado['is_relevant_gt'].astype(bool)).sum()}")
print(f"\nDistribución N2:")
print(df_anotado["procedures_gt"].value_counts().to_string())


Ground truth: 100 registros
Relevantes: 74 | No relevantes: 26

Distribución N2:
procedures_gt
AAP,AAC        16
AAP,AAC,DUP    12
AAU             8
IIA             8
DIA             6
IAE             6
AAI             5
DIA,AAI         4
AAP,DIA         3
AAP,AAC,DIA     2
AAP             2
AAC,AAP,DUP     1
DUP             1


---

##  4. Agente base

El agente Pydantic AI recibe una descripción de boletín y devuelve un `ClassifierOutput` validado. Se compara en dos configuraciones:

- **Baseline**: input = `description` + `bulletin`. El LLM infiere `act_type` desde el texto.
- **+N1**: input = `description` + `bulletin` + `act_type` pre-computado. El LLM lo usa como contexto.

In [ ]:
SYSTEM_PROMPT = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes pertenecen al universo de autorizaciones ambiental-energéticas (~3.7% del corpus).
El resto (RRHH, contratos, subvenciones, urbanismo...) son is_relevant=False.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental - resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa - valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción - permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada - equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental - evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada - permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico - aplica a planes y programas |
| DUP | Declaración de Utilidad Pública - reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo

## Reglas críticas
1. is_relevant=True SOLO si identificas al menos un procedimiento N2
2. AAU ≠ DIA - son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA - el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA - solo es DIA si menciona explícitamente "declaración de impacto ambiental"
7. IAE aplica a planes y programas, no a proyectos individuales
8. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
9. Las denegaciones y desistimientos heredan el tipo del procedimiento denegado
10. Las modificaciones heredan los procedimientos del acto modificado
11. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta
""".strip()

In [ ]:
agent = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT,
)



✓ Agente construido


`clasificar()` - función de inferencia unitaria

Recibe la descripción de un registro y devuelve un `ClassifierOutput` validado.

**Flujo interno:**

```
bulletin + description
    │
    ├─► get_ambito(bulletin)        → "estatal" / "autonómico" / "local"
    │
    ├─► [si use_n1_context=True]
    │       inferir_act_type()      → añade "Tipo de acto: resolución" al mensaje
    │
    ├─► agent.run(user_msg)         → llamada al LLM (async, espera respuesta)
    │
    └─► result.output              → ClassifierOutput ya validado por Pydantic
```

**Parámetros:**
- `description` - texto del boletín (ya con `html.unescape` aplicado)
- `bulletin` - código del boletín en minúsculas (`"boja"`, `"boe"`...)
- `use_n1_context` - si `True`, incluye el `act_type` pre-computado como contexto extra

**Configuraciones de experimento:**

| `use_n1_context` | Experimento |
|-----------------|-------------|
| `False` | Exp 1 - Baseline / Exp 2 - Prompt v2 |
| `True` | Exp 3 - Ablación +N1 |

In [13]:
async def clasificar(description: str, bulletin: str, use_n1_context: bool = False) -> ClassifierOutput:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    result = await agent.run(user_msg)
    return result.output

In [ ]:
# Casos cualitativos - 5 ejemplos de validación
casos = [
    ("Resolución de 12 de marzo de 2025, de la Dirección General de Calidad y Evaluación "
     "Ambiental, por la que se formula la declaración de impacto ambiental del proyecto "
     "Planta Solar Fotovoltaica Los Llanos, en la provincia de Cáceres.", "doe"),
    ("Resolución de 5 de febrero de 2025, de la Dirección General de Política Energética, "
     "por la que se otorga autorización administrativa previa y de construcción para el "
     "Parque Eólico Sierra Norte, de 48 MW, en Salamanca.", "boe"),
    ("Resolución de 18 de enero de 2025, de la Delegación Territorial de Medio Ambiente, "
     "por la que se otorga autorización ambiental unificada para la planta de biogás "
     "Valdecorneja, en Ávila.", "boja"),
    ("Resolución de 3 de marzo de 2025, de la Universidad de Salamanca, por la que se "
     "convoca concurso-oposición para cubrir plazas de profesor ayudante doctor.", "bocyl"),
    ("Resolución de 21 de febrero de 2025, de la Dirección General de Medio Natural, "
     "por la que se formula el informe de impacto ambiental del proyecto de línea "
     "eléctrica subterránea de 132 kV en Zaragoza.", "boa"),
]

EXPECTED = [
    {"procedures": {"DIA"}, "technologies": {"fotovoltaica"}, "relevant": True},
    {"procedures": {"AAP","AAC"}, "technologies": {"eólica"}, "relevant": True},
    {"procedures": {"AAU"}, "technologies": {"biogás_biometano"}, "relevant": True},
    {"procedures": set(), "technologies": set(), "relevant": False},
    {"procedures": {"IIA"}, "technologies": {"línea_eléctrica"}, "relevant": True},
]

print("Validación cualitativa - 5 casos\n")
aciertos = 0
for i, (desc, bul) in enumerate(casos, 1):
    r = await clasificar(desc, bul)
    pred_proc = set(p.value for p in r.procedures)
    pred_tech = set(t.value for t in r.technologies)
    exp = EXPECTED[i-1]
    ok = (r.is_relevant == exp["relevant"] and pred_proc == exp["procedures"])
    aciertos += ok
    mark = "✅" if ok else "❌"
    print(f"{mark} Caso {i} | is_relevant={r.is_relevant} | procedures={pred_proc} | technologies={pred_tech}")
    if not ok:
        print(f"   Esperado: relevant={exp['relevant']} procedures={exp['procedures']}")
print(f"\nResultado: {aciertos}/5 correctos")

Validación cualitativa — 5 casos

✅ Caso 1 | is_relevant=True | procedures={'DIA'} | technologies={'fotovoltaica'}
✅ Caso 2 | is_relevant=True | procedures={'AAP', 'AAC'} | technologies={'eólica'}
✅ Caso 3 | is_relevant=True | procedures={'AAU'} | technologies={'biogás_biometano'}
✅ Caso 4 | is_relevant=False | procedures=set() | technologies=set()
✅ Caso 5 | is_relevant=True | procedures={'IIA'} | technologies={'línea_eléctrica'}

Resultado: 5/5 correctos


---

##  5. Experimento 1 - Baseline

Configuración zero-shot sin contexto N1. El LLM recibe solo `description` + `bulletin` y debe inferir todos los campos por sí solo.

**Modelo**: Qwen 3.5 9B (thinking desactivado)
**Registros**: 100 (muestra estratificada del ground truth)
**Concurrencia**: 1 (DEC-009: modelos locales procesan secuencialmente)

In [14]:
from tqdm.asyncio import tqdm_asyncio


# ── clasificar_async ──────────────────────────────────────────────────────────
# Versión async de clasificar() orientada a ejecución en lote.
# Devuelve un dict plano listo para añadir como fila al CSV de resultados.
# Si el LLM falla (timeout, schema inválido tras retries...) captura el error
# y devuelve una fila de error en lugar de interrumpir todo el experimento.
async def clasificar_async(
    description: str,
    bulletin: str,
    use_n1_context: bool = False,
) -> dict:
    # Construir el mensaje de usuario (igual que en clasificar())
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"

    try:
        result = await agent.run(user_msg)
        output = result.output

        # Serializar a dict: los enums se convierten a string, las listas a JSON
        return {
            "is_relevant_pred":  output.is_relevant,
            "act_type_pred":     output.act_type.value,
            "procedures_pred":   json.dumps([p.value for p in output.procedures],   ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence":        output.confidence,
            "reasoning":         output.reasoning,
        }

    except Exception as e:
        # Fila de error — no rompe el bucle, se puede identificar después
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


# ── run_experiment ────────────────────────────────────────────────────────────
# Ejecuta clasificar_async sobre cada fila del DataFrame de entrada.
# - concurrency: cuántas llamadas al LLM van en paralelo (1 para modelos locales)
# - output_path: el CSV se guarda al terminar; si se interrumpe se pierde
async def run_experiment(
    df_input: pd.DataFrame,
    use_n1_context: bool = False,
    concurrency: int = 1,
    output_path: str = "../results/experiment.csv",
) -> pd.DataFrame:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # Semáforo: limita cuántas corrutinas corren a la vez
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async(row["description"], row["bulletin"], use_n1_context)
            # Combinar columnas originales + predicciones en una sola fila
            return {**row.to_dict(), **pred}

    # Lanzar todas las tareas y esperar con barra de progreso
    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando")

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)

    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


In [ ]:
# Ejecutar Experimento 1 - Baseline 
# Requiere LM Studio activo con el modelo cargado

df_exp1 = await run_experiment(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp1_baseline_qwen9b.csv",
)

print(df_exp1[["grupo_muestreo", "procedures_pred", "confidence"]].head(10))

Clasificando: 100%|██████████| 100/100 [1:32:42<00:00, 55.63s/it]


✓ Guardado en ../results/exp1_baseline_qwen9b.csv | Errores: 0/100
  grupo_muestreo        procedures_pred  confidence
0            AAC         ["AAP", "AAC"]        0.98
1            AAC  ["AAP", "AAC", "DUP"]        0.98
2            AAC  ["AAP", "AAC", "DIA"]        0.95
3            AAC         ["AAP", "AAC"]        0.98
4            AAC         ["AAP", "AAC"]        1.00
5            AAC         ["AAP", "AAC"]        0.98
6            AAC         ["AAP", "AAC"]        0.98
7            AAC  ["AAP", "AAC", "DUP"]        0.98
8            AAC  ["AAP", "AAC", "DUP"]        0.98
9            AAC  ["AAP", "AAC", "DUP"]        0.95


---

##  6. Análisis de errores - Baseline

Evaluamos las predicciones del Experimento 1 contra las etiquetas manuales del ground truth. El objetivo es identificar patrones de error sistemáticos que guíen la mejora del prompt en  7.

In [15]:
def parse_labels(value) -> set:
    if pd.isna(value) or str(value).strip() in ("", "nan"): return set()
    v = str(value).strip()
    if v.startswith("["):
        try: return set(json.loads(v))
        except: pass
    return set(x.strip() for x in v.split(",") if x.strip())


def compute_metrics(df_eval):
    """Calcula métricas completas sobre el DataFrame mergeado."""
    y_true = df_eval["is_relevant_gt"].astype(bool)
    y_pred = df_eval["is_relevant_pred"].fillna(False).astype(bool)

    # is_relevant
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
    fn=((y_true)&(~y_pred)).sum(); tn=((~y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0

    print(f"── is_relevant  P={p:.3f}  R={r:.3f}  F1={f1_rel:.3f}  TP={tp} FP={fp} FN={fn} TN={tn}\n")

    # N2
    N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]
    print(f"{'Label':<8} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5} {'TP':>4} {'FP':>4} {'FN':>4}")
    print("─" * 55)
    macro = 0
    for label in N2:
        yt = df_eval.apply(lambda r: label in parse_labels(r["procedures_gt"]), axis=1)
        yp = df_eval.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0
        r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        f12=2*p2*r2/(p2+r2) if p2+r2>0 else 0
        sup=yt.sum(); macro+=f12
        print(f"{label:<8} {p2:>6.3f} {r2:>6.3f} {f12:>6.3f} {sup:>5} {tp2:>4} {fp2:>4} {fn2:>4}")
    print("─" * 55)
    print(f"{'Macro-F1':<8} {macro/len(N2):>6.3f}")

    # Exact match
    exact = df_eval.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    rel = y_true
    print(f"\nExact match (relevantes): {exact[rel].mean():.3f}  ({exact[rel].sum()}/{rel.sum()})")
    print(f"Exact match (todos):      {exact.mean():.3f}  ({exact.sum()}/{len(df_eval)})")

    # Confidence
    conf = df_eval["confidence"].dropna()
    print(f"\nConfianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return exact, rel

In [16]:
# Cargar resultados del Experimento 1
df_exp1 = pd.read_csv("../results/exp1_baseline_qwen9b.csv")

# Merge con ground truth por descripción
df_eval1 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp1[["description","is_relevant_pred","act_type_pred","procedures_pred",
              "technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print(f"Registros evaluados: {len(df_eval1)}")
print(f"Predicciones válidas: {df_eval1['is_relevant_pred'].notna().sum()}\n")

exact1, rel1 = compute_metrics(df_eval1)

Registros evaluados: 100
Predicciones válidas: 100

── is_relevant  P=1.000  R=0.770  F1=0.870  TP=57 FP=0 FN=17 TN=26

Label         P      R     F1   Sup   TP   FP   FN
───────────────────────────────────────────────────────
DIA       1.000  0.800  0.889    15   12    0    3
AAP       1.000  0.917  0.957    36   33    0    3
AAC       1.000  0.935  0.967    31   29    0    2
AAU       1.000  0.875  0.933     8    7    0    1
IIA       1.000  0.375  0.545     8    3    0    5
AAI       1.000  0.556  0.714     9    5    0    4
IAE       1.000  0.167  0.286     6    1    0    5
DUP       1.000  0.857  0.923    14   12    0    2
───────────────────────────────────────────────────────
Macro-F1  0.777

Exact match (relevantes): 0.730  (54/74)
Exact match (todos):      0.800  (80/100)

Confianza: media=0.964  min=0.000  max=1.000


In [17]:
# Análisis detallado de errores N2
print("── Errores N2 en registros relevantes ──────────────────────────────\n")
errores1 = df_eval1[~exact1 & rel1]
print(f"Total errores: {len(errores1)}\n")

for _, row in errores1.iterrows():
    gt = parse_labels(row["procedures_gt"])
    pred = parse_labels(row["procedures_pred"])
    missing = sorted(gt - pred)
    extra = sorted(pred - gt)
    print(f"ID {row['id']} | {str(row.get('bulletin','')[:10])}")
    print(f"  GT  : {sorted(gt)}")
    print(f"  PRED: {sorted(pred)}")
    if missing: print(f"  Falta : {missing}")
    if extra:   print(f"  Sobra : {extra}")
    print(f"  Desc: {str(row['description'])[:100]}...")
    print(f"  Razón: {str(row['reasoning'])[:120]}")
    print()

── Errores N2 en registros relevantes ──────────────────────────────

Total errores: 20

ID 11 | 
  GT  : ['AAI']
  PRED: []
  Falta : ['AAI']
  Desc: Resolución de 11/12/2024, de la Dirección General de Calidad Ambiental, por la que se consideran no ...
  Razón: El texto menciona una "Resolución" que considera no sustanciales modificaciones y modifica la Resolución de 13/02/2019 s

ID 14 | 
  GT  : ['AAI']
  PRED: []
  Falta : ['AAI']
  Desc: Anuncio por el que se hace pública la Resolución de la Consejería de Transición Ecológica, Industria...
  Razón: El texto menciona "autorización ambiental integrada simplificada" para una instalación de "fabricación de hormigones fre

ID 32 | 
  GT  : ['AAC', 'AAP', 'DUP']
  PRED: []
  Falta : ['AAC', 'AAP', 'DUP']
  Desc: RESOLUCIÓN de 12 de marzo de 2025, del Servicio Territorial de Industria, Energía y Minas de Valenci...
  Razón: El texto indica una "resolución" por la que se "da por desistido" el expediente de autorización administrativa pre

---

##  7. Prompt v2 - Mejora basada en errores

Los cambios respecto al Prompt v1 se agrupan en tres categorías:

**Cambios consolidados** - basados en errores sistemáticos del Experimento 1:
- Regla explícita AAI+DIA: cuando una resolución formula DIA y otorga AAI en el mismo acto → [DIA, AAI]
- Refuerzo de denegaciones/desistimientos con ejemplo concreto de DUP

**Cambios experimentales** - a validar con el tutor/equipo (PENDIENTE-002):
- Regla: "todo procedimiento N2 identificado implica is_relevant=True, independientemente del tipo de proyecto" → esto incluye IIA de sondeos de agua, IAE de planes urbanísticos, AAI de cementeras. El Baseline los marcaba como False coherentemente con una interpretación estricta del dominio energético. Aquí asumimos scope amplio para medir el impacto - si el cliente prefiere scope estricto, esta regla se elimina y las métricas del Baseline en IIA/IAE/AAI mejorarían.
- DIA de proyectos no energéticos (concentración parcelaria, agroturismo) → is_relevant=True

**Sin cambios** - reglas del v1 que funcionaron bien:
- AAP, AAC, AAU, DUP → F1 perfecto o casi perfecto, no tocar
- Combinaciones AAP+AAC y AAP+AAC+DUP → ya funcionan

In [ ]:
SYSTEM_PROMPT_V2 = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes son aquellas que contienen al menos un procedimiento N2.
El tipo de proyecto NO determina la relevancia - una IIA sobre un sondeo de agua,
una IAE sobre un plan urbanístico o una AAI sobre una cementera son igualmente relevantes.
Son is_relevant=False: RRHH, contratos, subvenciones, licitaciones, padrones fiscales,
convenios de transporte, telecomunicaciones, plantillas orgánicas.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental - resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa - valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción - permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada - equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental - evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada - permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico - aplica a planes y programas |
| DUP | Declaración de Utilidad Pública - reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo
Si el proyecto no es energético, technologies=[]

## Reglas críticas
1. is_relevant=True si y solo si identificas al menos un procedimiento N2 - independientemente del tipo de proyecto
2. AAU ≠ DIA - son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA - el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA - solo añadir DIA si el texto menciona EXPLÍCITAMENTE "declaración de impacto ambiental"
7. Cuando una resolución formula DIA Y otorga AAI en el mismo acto → [DIA, AAI]
8. IAE aplica a planes y programas, no a proyectos individuales
9. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
10. Denegaciones y desistimientos heredan el tipo del procedimiento - ejemplo: "se da por desistido el titular de AAP+AAC+DUP" → [AAP, AAC, DUP]
11. Las modificaciones heredan los procedimientos del acto modificado
12. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta
""".strip()

---

##  8. Experimento 2 - Prompt v2

Mismo modelo (Qwen 3.5 9B), mismos 100 registros, pero con el prompt mejorado. Comparamos con Experimento 1 para cuantificar la ganancia del prompt engineering.

In [21]:
# Agente v2 — mismo modelo, prompt mejorado
agent_v2 = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V2,
)


# ── clasificar_async_v2 / run_experiment_v2 ───────────────────────────────────
# Idénticas a las de §5 pero usan agent_v2 (prompt v2).
# Se definen por separado para poder ejecutar ambos experimentos en el mismo
# kernel sin sobreescribir las funciones del Experimento 1.
async def clasificar_async_v2(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_v2.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred":  output.is_relevant,
            "act_type_pred":     output.act_type.value,
            "procedures_pred":   json.dumps([p.value for p in output.procedures],   ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence":        output.confidence,
            "reasoning":         output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


async def run_experiment_v2(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_v2(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando v2")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


# ── Ejecutar Experimento 2 ────────────────────────────────────────────────────
df_exp2 = await run_experiment_v2(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp2_promptv2_qwen9b.csv",
)


Clasificando v2: 100%|██████████| 100/100 [1:31:11<00:00, 54.71s/it]


✓ Guardado en ../results/exp2_promptv2_qwen9b.csv | Errores: 2/100


In [ ]:
df_exp2 = pd.read_csv("../results/exp2_promptv2_qwen9b.csv")
df_eval2 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp2[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 2 - Prompt v2 ──────────────────────────")
exact2, rel2 = compute_metrics(df_eval2)

── Experimento 2 — Prompt v2 ──────────────────────────
── is_relevant  P=0.986  R=0.959  F1=0.973  TP=71 FP=1 FN=3 TN=25

Label         P      R     F1   Sup   TP   FP   FN
───────────────────────────────────────────────────────
DIA       0.929  0.867  0.897    15   13    1    2
AAP       1.000  0.944  0.971    36   34    0    2
AAC       1.000  0.935  0.967    31   29    0    2
AAU       1.000  1.000  1.000     8    8    0    0
IIA       1.000  1.000  1.000     8    8    0    0
AAI       0.875  0.778  0.824     9    7    1    2
IAE       1.000  1.000  1.000     6    6    0    0
DUP       1.000  1.000  1.000    14   14    0    0
───────────────────────────────────────────────────────
Macro-F1  0.957

Exact match (relevantes): 0.919  (68/74)
Exact match (todos):      0.930  (93/100)

Confianza: media=0.963  min=0.000  max=1.000


---

##  9. Experimento 3 - Ablación +N1

Añadimos el `act_type` pre-computado por reglas como contexto al prompt. Pregunta de investigación: **¿cuánto aporta saber la forma jurídica del documento para clasificar el procedimiento N2?**

Usamos Prompt v2 + contexto N1.

In [20]:
df_exp3 = await run_experiment_v2(
    df_anotado,
    use_n1_context=True,  # ← única diferencia
    concurrency=1,
    output_path="../results/exp3_promptv2_n1_qwen9b.csv",
)

NameError: name 'run_experiment_v2' is not defined

In [ ]:
df_exp3 = pd.read_csv("../results/exp3_promptv2_n1_qwen9b.csv")
df_eval3 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp3[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 3 - Prompt v2 + N1 ────────────────────")
exact3, rel3 = compute_metrics(df_eval3)

── Experimento 3 — Prompt v2 + N1 ────────────────────
── is_relevant  P=1.000  R=0.973  F1=0.986  TP=72 FP=0 FN=2 TN=26

Label         P      R     F1   Sup   TP   FP   FN
───────────────────────────────────────────────────────
DIA       1.000  0.933  0.966    15   14    0    1
AAP       1.000  0.972  0.986    36   35    0    1
AAC       1.000  0.968  0.984    31   30    0    1
AAU       1.000  1.000  1.000     8    8    0    0
IIA       1.000  1.000  1.000     8    8    0    0
AAI       1.000  0.667  0.800     9    6    0    3
IAE       1.000  0.833  0.909     6    5    0    1
DUP       1.000  1.000  1.000    14   14    0    0
───────────────────────────────────────────────────────
Macro-F1  0.956

Exact match (relevantes): 0.919  (68/74)
Exact match (todos):      0.940  (94/100)

Confianza: media=0.969  min=0.850  max=1.000


---

##  10. Experimento 4 - Few-shot

Añadimos ejemplos reales al prompt (one o two-shot por categoría difícil). Pregunta de investigación: **¿mejora la clasificación de los casos borde cuando el modelo tiene ejemplos concretos?**

Los ejemplos se seleccionan de los errores identificados en  6.

> ⚙️ **Pendiente**: implementar tras analizar los errores del Experimento 1.

In [ ]:
SYSTEM_PROMPT_V3 = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes son aquellas que contienen al menos un procedimiento N2.
El tipo de proyecto NO determina la relevancia - una IIA sobre un sondeo de agua,
una IAE sobre un plan urbanístico o una AAI sobre una cementera son igualmente relevantes.
Son is_relevant=False: RRHH, contratos, subvenciones, licitaciones, padrones fiscales,
convenios de transporte, telecomunicaciones, plantillas orgánicas.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental - resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa - valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción - permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada - equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental - evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada - permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico - aplica a planes y programas |
| DUP | Declaración de Utilidad Pública - reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo
Si el proyecto no es energético, technologies=[]

## Reglas críticas
1. is_relevant=True si y solo si identificas al menos uno de estos procedimientos:
   DIA, AAP, AAC, AAU, IIA, AAI, IAE o DUP - independientemente del tipo de proyecto
2. AAU ≠ DIA - son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA - el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA - solo añadir DIA si el texto menciona EXPLÍCITAMENTE "declaración de impacto ambiental"
7. Cuando una resolución formula DIA Y otorga o modifica AAI en el mismo acto → [DIA, AAI]
8. IAE aplica a planes y programas, no a proyectos individuales
9. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
10. Denegaciones y desistimientos heredan el tipo del procedimiento - ejemplo: "se da por desistido el titular de AAP+AAC+DUP" → [AAP, AAC, DUP]
11. Las modificaciones heredan los procedimientos del acto modificado
12. Los ANUNCIOS de información pública sobre solicitudes son tan relevantes como las resoluciones - etiquetar según los procedimientos que mencionen
13. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta

## Ejemplos

### Ejemplo 1 - AAI simplificada (is_relevant=True aunque el proyecto no sea energético)
Descripción: «Anuncio por el que se hace pública la Resolución de la Consejería de Transición
Ecológica, Industria y Comercio, de otorgamiento de la autorización ambiental integrada
simplificada a la instalación de "fabricación de hormigones frescos" del titular Cementos
Secil, S.L.U., ubicada en polígono industrial La Curiscada (Tineo).»
→ is_relevant=True | procedures=[AAI] | technologies=[]
Razón: "autorización ambiental integrada simplificada" es una variante de AAI → relevante aunque sea industria del cemento.

### Ejemplo 2 - DIA + AAI en el mismo acto
Descripción: «Resolución de la Directora General de Armonización Urbanística y Evaluación
ambiental por la que se formula la declaración de impacto ambiental de la modificación
sustancial de la AAI IPPC 02/2015 centro de recepción y pretratamiento de hidrocarburos,
aceites usados y aguas aceitosas en el dique del Oeste a Palma.»
→ is_relevant=True | procedures=[DIA, AAI] | technologies=[petróleo]
Razón: "formula la declaración de impacto ambiental" → DIA. "modificación sustancial de la AAI" → AAI. Ambos en el mismo acto.

### Ejemplo 3 - Anuncio de información pública con AAP+AAC+DUP
Descripción: «Anuncio de 03/01/2025, de la Delegación Provincial de Desarrollo Sostenible
de Cuenca, sobre información pública de la solicitud de autorización administrativa previa,
aprobación del proyecto de ejecución y reconocimiento en concreto de utilidad pública
de la instalación eléctrica de alta tensión.»
→ is_relevant=True | procedures=[AAP, AAC, DUP] | technologies=[línea_eléctrica]
Razón: Los anuncios de solicitud son relevantes. "autorización administrativa previa" → AAP. "aprobación del proyecto de ejecución" → AAC. "reconocimiento en concreto de utilidad pública" → DUP.
""".strip()

In [ ]:
# ── Experimento 4 - Few-shot (Prompt v3) ─────────────────────────────────────

agent_v3 = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V3,
)

async def clasificar_async_v3(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_v3.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred": output.is_relevant,
            "act_type_pred": output.act_type.value,
            "procedures_pred": json.dumps([p.value for p in output.procedures], ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence": output.confidence,
            "reasoning": output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


async def run_experiment_v3(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_v3(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando v3")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


df_exp4 = await run_experiment_v3(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp4_fewshot_qwen9b.csv",
)

Clasificando v3: 100%|██████████| 100/100 [1:45:41<00:00, 63.42s/it]


✓ Guardado en ../results/exp4_fewshot_qwen9b.csv | Errores: 1/100


In [ ]:
df_exp4 = pd.read_csv("../results/exp4_fewshot_qwen9b.csv")
df_eval4 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp4[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 4 - Few-shot (Prompt v3) ──────────────────")
exact4, rel4 = compute_metrics(df_eval4)

── Experimento 4 — Few-shot (Prompt v3) ──────────────────
── is_relevant  P=1.000  R=0.986  F1=0.993  TP=73 FP=0 FN=1 TN=26

Label         P      R     F1   Sup   TP   FP   FN
───────────────────────────────────────────────────────
DIA       0.933  0.933  0.933    15   14    1    1
AAP       0.972  0.972  0.972    36   35    1    1
AAC       1.000  0.968  0.984    31   30    0    1
AAU       1.000  1.000  1.000     8    8    0    0
IIA       1.000  1.000  1.000     8    8    0    0
AAI       1.000  0.778  0.875     9    7    0    2
IAE       1.000  1.000  1.000     6    6    0    0
DUP       1.000  1.000  1.000    14   14    0    0
───────────────────────────────────────────────────────
Macro-F1  0.971

Exact match (relevantes): 0.946  (70/74)
Exact match (todos):      0.960  (96/100)

Confianza: media=0.977  min=0.950  max=1.000


---

##  11. Comparativa de modelos

Repetimos el mejor experimento (Prompt v2 + configuración óptima) con Gemma 4 4B.
Pregunta de investigación: **¿cuánto se pierde en F1 al usar un modelo 4B vs 9B?**

Para cambiar al Gemma 4B: en  0, cambiar `LM_STUDIO_MODEL = "gemma-4-4b-it"` y reiniciar el kernel.

> ⚙️ **Pendiente**: ejecutar cuando Gemma 4B esté descargado en LM Studio.

In [ ]:
# ── Experimento 5 — Gemma 4 4B (Prompt v2) ──────────────────────────────────
# Mismo prompt v2, mismo ground truth — solo cambia el modelo.
# Permite comparar Qwen 9B vs Gemma 4B con todas las demás variables fijas.

from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

agent_gemma = Agent(
    model_gemma,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V2,
)


# ── clasificar_async_gemma / run_experiment_gemma ─────────────────────────────
# Idénticas a las versiones anteriores pero usan agent_gemma.
async def clasificar_async_gemma(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_gemma.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred":  output.is_relevant,
            "act_type_pred":     output.act_type.value,
            "procedures_pred":   json.dumps([p.value for p in output.procedures],   ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence":        output.confidence,
            "reasoning":         output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


async def run_experiment_gemma(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_gemma(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando Gemma 4B")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


In [ ]:
# Ejecutar - primero para el modelo con Prompt v2
df_exp5 = await run_experiment_gemma(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp5_promptv2_gemma4b.csv",
)

Clasificando Gemma 4B: 100%|██████████| 100/100 [5:12:12<00:00, 187.33s/it] 


✓ Guardado en ../results/exp5_promptv2_gemma4b.csv | Errores: 3/100


In [ ]:
df_exp5 = pd.read_csv("../results/exp5_promptv2_gemma4b.csv")
df_eval5 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp5[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 5 - Gemma 4 4B · Prompt v2 ────────────────")
exact5, rel5 = compute_metrics(df_eval5)

── Experimento 5 — Gemma 4 4B · Prompt v2 ────────────────
── is_relevant  P=0.986  R=0.973  F1=0.980  TP=72 FP=1 FN=2 TN=25

Label         P      R     F1   Sup   TP   FP   FN
───────────────────────────────────────────────────────
DIA       1.000  0.933  0.966    15   14    0    1
AAP       1.000  0.972  0.986    36   35    0    1
AAC       1.000  0.968  0.984    31   30    0    1
AAU       1.000  1.000  1.000     8    8    0    0
IIA       0.875  0.875  0.875     8    7    1    1
AAI       0.889  0.889  0.889     9    8    1    1
IAE       1.000  1.000  1.000     6    6    0    0
DUP       1.000  1.000  1.000    14   14    0    0
───────────────────────────────────────────────────────
Macro-F1  0.962

Exact match (relevantes): 0.946  (70/74)
Exact match (todos):      0.950  (95/100)

Confianza: media=0.960  min=0.100  max=1.000


---

##  12. Tabla resumen - Comparativa de experimentos

In [18]:
import pandas as pd
import json
import os

def parse_labels(value):
    if pd.isna(value) or str(value).strip() in ('', 'nan'): return set()
    v = str(value).strip()
    if v.startswith('['):
        try: return set(json.loads(v))
        except: pass
    return set(x.strip() for x in v.split(',') if x.strip())

# Cargar ground truth directamente
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_100_anotado.csv")

N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]

experimentos = [
    ("Exp 1 · Baseline zero-shot",   "Qwen 3.5 9B", "Zero-shot",   "../results/exp1_baseline_qwen9b.csv"),
    ("Exp 2 · Prompt v2",            "Qwen 3.5 9B", "Zero-shot",   "../results/exp2_promptv2_qwen9b.csv"),
    ("Exp 3 · v2 + N1",              "Qwen 3.5 9B", "+N1 context", "../results/exp3_promptv2_n1_qwen9b.csv"),
    ("Exp 4 · Few-shot (v3)",         "Qwen 3.5 9B", "Few-shot",    "../results/exp4_fewshot_qwen9b.csv"),
    ("Exp 5 · Gemma 4B · Prompt v2", "Gemma 4 4B",  "Zero-shot",   "../results/exp5_promptv2_gemma4b.csv"),
]

rows = []
for nombre, modelo, config, path in experimentos:
    if not os.path.exists(path):
        print(f"⚠️  {nombre}: archivo no encontrado")
        continue

    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_r[["description","is_relevant_pred","procedures_pred","confidence","reasoning"]],
        on="description", how="left"
    )

    y_true = df_e["is_relevant_gt"].astype(bool)
    y_pred = df_e["is_relevant_pred"].fillna(False).astype(bool)
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
    fn=((y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0
    r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0

    macro = 0
    for label in N2:
        yt = df_e.apply(lambda r: label in parse_labels(r["procedures_gt"]), axis=1)
        yp = df_e.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0
        r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        macro += 2*p2*r2/(p2+r2) if p2+r2>0 else 0

    exact = df_e.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    errores = df_e["reasoning"].astype(str).str.startswith("ERROR").sum()
    conf_mean = df_e["confidence"].mean()
    conf_min = df_e["confidence"].min()

    rows.append({
        "Experimento": nombre,
        "Modelo": modelo,
        "Config": config,
        "is_rel F1": f"{f1_rel:.3f}",
        "Macro-F1": f"{macro/len(N2):.3f}",
        "Exact(rel)": f"{exact[y_true].mean():.3f}",
        "Exact(all)": f"{exact.mean():.3f}",
        "Conf media": f"{conf_mean:.3f}",
        "Conf min": f"{conf_min:.3f}",
        "Errores fmt": errores,
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv("../results/tabla_resumen_experimentos.csv", index=False)
print("\n✓ Guardado en ../results/tabla_resumen_experimentos.csv")

                 Experimento      Modelo      Config is_rel F1 Macro-F1 Exact(rel) Exact(all) Conf media Conf min  Errores fmt
  Exp 1 · Baseline zero-shot Qwen 3.5 9B   Zero-shot     0.870    0.777      0.730      0.800      0.964    0.000            0
           Exp 2 · Prompt v2 Qwen 3.5 9B   Zero-shot     0.973    0.957      0.919      0.930      0.963    0.000            2
             Exp 3 · v2 + N1 Qwen 3.5 9B +N1 context     0.986    0.956      0.919      0.940      0.969    0.850            0
       Exp 4 · Few-shot (v3) Qwen 3.5 9B    Few-shot     0.993    0.971      0.946      0.960      0.977    0.950            1
Exp 5 · Gemma 4B · Prompt v2  Gemma 4 4B   Zero-shot     0.980    0.962      0.946      0.950      0.960    0.100            3

✓ Guardado en ../results/tabla_resumen_experimentos.csv


---

##  13. Calibración de confianza

Analizamos si el campo `confidence` es un predictor real de calidad. **Hipótesis**: los registros con `confidence < 0.8` deberían tener peor F1 que los de `confidence > 0.95`.

Si el modelo es sobreconfiante (todos los valores entre 0.95-1.0), el campo `confidence` no sirve como filtro práctico - esto es un hallazgo relevante para la memoria.

In [ ]:
# Comparativa de calibración entre modelos
print(" Calibración comparativa ")
print()

calibracion = [
    ("Qwen 3.5 9B · Baseline",  "../results/exp1_baseline_qwen9b.csv"),
    ("Qwen 3.5 9B · Prompt v2", "../results/exp2_promptv2_qwen9b.csv"),
    ("Qwen 3.5 9B · +N1",       "../results/exp3_promptv2_n1_qwen9b.csv"),
    ("Qwen 3.5 9B · Few-shot",  "../results/exp4_fewshot_qwen9b.csv"),
    ("Gemma 4 4B · Prompt v2",  "../results/exp5_promptv2_gemma4b.csv"),
]

print(f"{'Experimento':<30} {'Media':>7} {'Min':>7} {'>=0.95':>8} {'Predictor?':>12}")
print("─" * 70)

for nombre, path in calibracion:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_r[["description","is_relevant_pred","procedures_pred","confidence"]],
        on="description", how="left"
    )
    df_e = df_e[df_e["confidence"].notna()]
    df_e["exact"] = df_e.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )

    conf = df_e["confidence"]
    alta = df_e[df_e["confidence"] >= 0.95]["exact"].mean()
    baja_n = (df_e["confidence"] < 0.95).sum()
    baja_acc = df_e[df_e["confidence"] < 0.95]["exact"].mean() if baja_n > 0 else float("nan")
    sobreconf = (conf >= 0.95).mean()

    # Es predictor si hay diferencia significativa entre alta y baja confianza
    if baja_n >= 3 and not pd.isna(baja_acc):
        predictor = "✅ Sí" if (alta - baja_acc) > 0.1 else "❌ No"
    else:
        predictor = "- insuf."

    print(f"{nombre:<30} {conf.mean():>7.3f} {conf.min():>7.3f} {sobreconf:>7.1%}  {predictor:>12}")

print()
print("Conclusión: solo Gemma 4B tiene distribución de confianza suficientemente")
print("dispersa para usarse como señal de revisión manual.")

── Calibración comparativa ──────────────────────────────────

Experimento                      Media     Min   >=0.95   Predictor?
──────────────────────────────────────────────────────────────────────
Qwen 3.5 9B · Baseline           0.964   0.000   98.0%      — insuf.
Qwen 3.5 9B · Prompt v2          0.963   0.000   99.0%      — insuf.
Qwen 3.5 9B · +N1                0.969   0.850   98.0%      — insuf.
Qwen 3.5 9B · Few-shot           0.977   0.950  100.0%      — insuf.
Gemma 4 4B · Prompt v2           0.960   0.100   97.9%      — insuf.

Conclusión: solo Gemma 4B tiene distribución de confianza suficientemente
dispersa para usarse como señal de revisión manual.


---

##  14. Conclusiones y trabajo futuro

### Hallazgos principales

> *Completar tras ejecutar todos los experimentos*

### Limitaciones

1. **Ground truth Opción A**: el dataset de 100 registros fue anotado manualmente sin revisión cruzada. La Opción B (anotación rigurosa con múltiples anotadores) queda como trabajo futuro.
2. **Modelo local limitado**: Qwen 3.5 9B con 16GB RAM es el modelo más grande ejecutable en el hardware disponible. Los experimentos con modelos frontera (Gemini 2.5 Pro) se limitan a mini-muestras por rate limits.
3. **Sobreconfianza**: el campo `confidence` no es un predictor fiable de calidad en modelos locales - todos los valores tienden a 0.95-1.0.

### Trabajo futuro - v2

| Tarea | Descripción |
|-------|-------------|
| Ampliar corpus | Extender a las 18 familias y 92 procedimientos del N2 completo |
| Ground truth riguroso | Opción B: anotación manual con revisión cruzada |
| Nuevas tecnologías N3 | `industria_ippc`, `infraestructura_hidrica`, `ordenacion_territorial`, `turismo_edificacion` |
| N1 para RRHH | LLM zero-shot para registros sin tipo de acto explícito |
| Modelos frontera | Evaluación completa con Gemini 2.5 Pro / GPT-4o |